In [22]:
import pandas as pd
import os

# Full Table

In [23]:
results_dir = "/hdd/ivny/results"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

methods_names = {
    "dist_linguistic_confidence": "Ling. Conf.",
    "dist_semantic_uncertainty": "Semantic Unc.",
    "dist_lnll": "Token Prob"
}

dataset_map = {
    "mmlu": "MMLU",
    "squadv2": "SQuADv2.0",
    "trivia_qa": "TriviaQA"
}

model_name_map = {
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B-Inst.",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B-Inst.",
    "Qwen2.5-7B-Instruct": "Qwen2.5-7B-Inst.",
    "Qwen3-8B": "Qwen3-8B-Inst.",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B-Inst.",
    "gpt-oss-20b": "GPT-OSS-20B",
}

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, estimation_method, _, model_name, _ = leaf_dir.split("/")
    csv_data = pd.read_csv(os.path.join(leaf_dir, "eval_metrics.csv")).to_dict(orient="records")[0]
    df_dict = {
        "Dataset": dataset_map[dataset_name],
        "Est. Method": methods_names[estimation_method],
        "Model": model_name_map[model_name],
        **csv_data
    }
    all_records.append(df_dict)

full_results_df = pd.DataFrame(all_records).round(3)
full_results_df.rename(columns={"accuracy": "Acc", "generalised_ece": r"ECE$_{gen}$", "dECE_point_mass": r"ECE$_{pm}$", "AUROC_point_mass": r"AUROC$_{pm}$"}, inplace=True)
full_results_df

,Dataset,Est. Method,Model,Acc,ECE$_{gen}$,dECE,ECE$_{pm}$,dAUROC,AUROC$_{pm}$
0,MMLU,Ling. Conf.,Llama-3.1-8B-Inst.,0.626,0.124,0.126,0.093,0.609,0.645
1,MMLU,Ling. Conf.,Llama-3-8B-Inst.,0.619,0.190,0.191,0.166,0.557,0.583
2,MMLU,Ling. Conf.,Qwen2.5-7B-Inst.,0.672,0.142,0.145,0.069,0.566,0.592
3,MMLU,Ling. Conf.,Qwen3-8B-Inst.,0.749,0.146,0.147,0.062,0.562,0.595
4,MMLU,Ling. Conf.,Mistral-7B-Inst.,0.566,0.277,0.277,0.213,0.526,0.529
5,MMLU,Ling. Conf.,GPT-OSS-20B,0.774,0.157,0.160,0.073,0.562,0.576
6,MMLU,Semantic Unc.,Llama-3.1-8B-Inst.,0.609,0.234,0.272,0.227,0.729,0.728
7,MMLU,Semantic Unc.,Llama-3-8B-Inst.,0.595,0.254,0.264,0.249,0.709,0.708
8,MMLU,Semantic Unc.,Qwen2.5-7B-Inst.,0.708,0.273,0.302,0.272,0.547,0.547
9,MMLU,Semantic Unc.,Qwen3-8B-Inst.,0.693,0.271,0.303,0.270,0.586,0.586


# Average Aggregated Table

In [27]:
# Aggregate
latex_output = (
    full_results_df
    .drop(columns=["Dataset", "Acc"])
    .groupby(["Est. Method", "Model"])
    .mean()
    .round(3)
    .to_latex(
        multirow=True,
        float_format="%.3f",
        escape=False,
        column_format="l p{2.8cm} rrrrr"  # <-- THIS controls wrapping
    )
)

print(latex_output)

\begin{tabular}{l p{2.8cm} rrrrr}
\toprule
 &  & ECE$_{gen}$ & dECE & ECE$_{pm}$ & dAUROC & AUROC$_{pm}$ \\
Est. Method & Model &  &  &  &  &  \\
\midrule
\multirow[t]{6}{*}{Ling. Conf.} & GPT-OSS-20B & 0.255 & 0.256 & 0.222 & 0.617 & 0.648 \\
 & Llama-3-8B-Inst. & 0.198 & 0.199 & 0.170 & 0.578 & 0.606 \\
 & Llama-3.1-8B-Inst. & 0.144 & 0.146 & 0.125 & 0.659 & 0.695 \\
 & Mistral-7B-Inst. & 0.276 & 0.277 & 0.249 & 0.564 & 0.604 \\
 & Qwen2.5-7B-Inst. & 0.184 & 0.186 & 0.155 & 0.605 & 0.642 \\
 & Qwen3-8B-Inst. & 0.239 & 0.239 & 0.207 & 0.565 & 0.620 \\
\cline{1-7}
\multirow[t]{6}{*}{Semantic Unc.} & GPT-OSS-20B & 0.207 & 0.251 & 0.197 & 0.667 & 0.668 \\
 & Llama-3-8B-Inst. & 0.262 & 0.269 & 0.252 & 0.649 & 0.652 \\
 & Llama-3.1-8B-Inst. & 0.241 & 0.262 & 0.230 & 0.662 & 0.664 \\
 & Mistral-7B-Inst. & 0.308 & 0.296 & 0.299 & 0.632 & 0.638 \\
 & Qwen2.5-7B-Inst. & 0.242 & 0.257 & 0.234 & 0.644 & 0.649 \\
 & Qwen3-8B-Inst. & 0.247 & 0.254 & 0.240 & 0.659 & 0.662 \\
\cline{1-7}
\multirow[t